# Constitutional AI: 헌법적 AI - 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습

- Tutorial ID: `expand-constitutional-ai`
- Tutorial: Constitutional AI: 헌법적 AI
- Section ID: `expand-constitutional-ai-code-2`
- Section: 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: RLAIF: AI 피드백으로 보상 모델 학습
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from openai import OpenAI
import json
import random

client = OpenAI()

# ── RLAIF 보상 모델 학습 데이터 생성 ──
PRINCIPLES = [
    "Choose the response that is more helpful and harmless.",
    "Choose the response that is more honest and transparent.",
    "Choose the response that is less biased and more fair.",
]

def generate_preference_pairs(prompt: str, n_pairs: int = 5):
    """RLAIF: AI가 헌법에 따라 응답 쌍의 선호도 판단"""
    pairs = []
    
        for _ in range(n_pairs):
        # 두 개의 응답 생성 (다른 temperature로)
        resp_a = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
                        temperature=0.8,
        ).choices[0].message.content
        
        resp_b = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
                        temperature=0.2,
        ).choices[0].message.content
        
        # AI가 헌법에 따라 선호도 판단 (RLAIF 핵심)
        judge_prompt = f"""You are an AI judge following these principles:
{chr(10).join(PRINCIPLES)}

Which response is better for the following prompt?

PROMPT: {prompt}

RESPONSE A: {resp_a}

RESPONSE B: {resp_b}

Answer ONLY "A" or "B" and briefly explain why."""

        judgment = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": judge_prompt}],
                        temperature=0,
        ).choices[0].message.content
        
        chosen = resp_a if judgment.strip().startswith("A") else resp_b
        rejected = resp_b if judgment.strip().startswith("A") else resp_a
        
        pairs.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "judgment": judgment,
        })
    
    return pairs

# ── 선호도 데이터 생성 ──
prompts = [
    "파이썬으로 리스트 정렬하는 방법 알려줘",
    "건강한 식단에 대해 조언해 줘",
    "기후변화의 주요 원인은 뭐야?",
]

all_pairs = []
for p in prompts:
    all_pairs.extend(generate_preference_pairs(p, n_pairs=3))

# ── DPO 학습용 데이터셋 저장 ──
dpo_dataset = [{
    "prompt": p["prompt"],
    "chosen": p["chosen"],
    "rejected": p["rejected"],
} for p in all_pairs]

with open("rlaif_preference_data.json", "w") as f:
    json.dump(dpo_dataset, f, indent=2, ensure_ascii=False)

print(f"RLAIF 선호도 데이터 {len(dpo_dataset)}쌍 생성 완료")
print(f"→ 이 데이터로 DPO/RLHF 학습 가능!")

# 샘플 출력
for pair in dpo_dataset[:2]:
    print(f"\n프롬프트: {pair['prompt']}")
    print(f"  선호됨: {pair['chosen'][:80]}...")
    print(f"  기각됨: {pair['rejected'][:80]}...")